# Milestone 1 — Reproduce the frame-semantic-transformer baseline

Confirms we can reproduce the upstream **`base`** model's published FrameNet 1.7
test-set F1 with our own eval harness, so later "we beat it" comparisons are
credible.

**Reported test F1 (base):** trigger `0.74`, frame `0.89`, args `0.75`.

This notebook is self-contained: it installs the vendored parser at its pinned
commit **with `--no-deps`** (skipping the fragile 2023 torch / PyTorch-Lightning
pins) and drives evaluation with a manual loop that calls the *upstream* scoring
functions verbatim — identical numbers, modern environment.

> **Before running:** set `Runtime → Change runtime type → GPU` (A100 / L4 ideal).

In [ ]:
!nvidia-smi

## 1. Install

`--no-deps` avoids downgrading Colab's CUDA-matched torch. We only need the
parser's source (for its data loaders + scoring functions) plus a modern
transformers/sentencepiece/nltk. PyTorch-Lightning is deliberately **not**
installed — our eval loop doesn't use it.

In [ ]:
# Vendored parser at the exact commit recorded in PROVENANCE.md
!pip install -q --no-deps "git+https://github.com/texturejc/frame-semantic-transformer.git@18cb3023bbb6df0c1b53a52182135c0c0132c073"
# Runtime deps (torch already present on the Colab GPU image)
!pip install -q "transformers>=4.30,<4.45" sentencepiece protobuf nltk

In [ ]:
import nltk
# FrameNet 1.7 corpus + WordNet (used by the inference loader / lexicon lookup)
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## 2. Load the model + build the test dataset

The upstream `base` model lives on the HF hub at revision `v0.2.0`.

In [ ]:
import time
from collections import defaultdict

import torch
from torch.utils.data import DataLoader
from transformers import T5ForConditionalGeneration, T5TokenizerFast

from frame_semantic_transformer.constants import MODEL_MAX_LENGTH, MODEL_REVISION
from frame_semantic_transformer.data.LoaderDataCache import LoaderDataCache
from frame_semantic_transformer.data.TaskSampleDataset import TaskSampleDataset
from frame_semantic_transformer.data.loaders.framenet17 import (
    Framenet17InferenceLoader,
    Framenet17TrainingLoader,
)
from frame_semantic_transformer.data.tasks_from_annotated_sentences import (
    tasks_from_annotated_sentences,
)
from frame_semantic_transformer.training.evaluate_batch import (
    TaskEvalResults, calc_eval_metrics, evaluate_batch,
)
from frame_semantic_transformer.training.TrainingModelWrapper import merge_metrics

MODEL = "base"      # or "small"
SPLIT = "test"      # or "dev"
BATCH_SIZE = 16

model_ref = f"chanind/frame-semantic-transformer-{MODEL}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = T5ForConditionalGeneration.from_pretrained(model_ref, revision=MODEL_REVISION).to(device)
tokenizer = T5TokenizerFast.from_pretrained(
    model_ref, revision=MODEL_REVISION, model_max_length=MODEL_MAX_LENGTH, legacy=False
)
print("loaded", model_ref, "on", device)

In [ ]:
inference_loader = Framenet17InferenceLoader(); inference_loader.setup()
loader_cache = LoaderDataCache(inference_loader)

training_loader = Framenet17TrainingLoader(); training_loader.setup()
sentences = (training_loader.load_test_data() if SPLIT == "test"
             else training_loader.load_validation_data())
tasks = tasks_from_annotated_sentences(sentences, loader_cache)
dataset = TaskSampleDataset(tasks, tokenizer, balance_tasks=False)
print(f"{len(sentences)} sentences -> {len(tasks)} tasks -> {len(dataset)} samples")

## 3. Evaluate

Manual loop = upstream `TrainingModelWrapper.test_step` without Lightning. Each
batch is scored by the upstream `evaluate_batch`; results merged with the
upstream `merge_metrics`; F1 via the upstream `calc_eval_metrics`.

In [ ]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=2)
merged = defaultdict(TaskEvalResults)
model.eval()
t0 = time.time()
with torch.no_grad():
    for i, batch in enumerate(loader):
        merged = merge_metrics([merged, evaluate_batch(model, tokenizer, batch, loader_cache)])
        if (i + 1) % 10 == 0 or (i + 1) == len(loader):
            print(f"batch {i + 1}/{len(loader)}", flush=True)
elapsed = time.time() - t0

In [ ]:
REPORTED = {"test": {"trigger_identification": 0.74, "frame_classification": 0.89, "args_extraction": 0.75},
            "dev":  {"trigger_identification": 0.78, "frame_classification": 0.91, "args_extraction": 0.78}}[SPLIT]

print(f"{'task':<24}{'precision':>10}{'recall':>10}{'f1':>10}{'reported':>10}")
print("-" * 64)
for task in ("trigger_identification", "frame_classification", "args_extraction"):
    if task not in merged:
        continue
    s = calc_eval_metrics(merged[task].scores)
    print(f"{task:<24}{s['precision']:>10.3f}{s['recall']:>10.3f}{s['f_score']:>10.3f}{REPORTED[task]:>10.2f}")
print("-" * 64)
print(f"{len(dataset)} samples in {elapsed:.1f}s ({1000*elapsed/max(len(dataset),1):.1f} ms/sample)")

## Interpreting

F1 within **~1 point** of the reported column = baseline reproduced ✅.
Copy the measured F1 into `MILESTONES.md`, and note the ms/sample — that's the
**speed number** the encoder model (Milestone 2) has to beat.

Small discrepancies (<1 pt) are expected from tokenizer/transformers version
drift vs. 2023. A large gap most often means the model revision or the FrameNet
corpus version is wrong — check both.